# Task 2 — Trích xuất Code Property Graph (CPG Parser) và Hợp đồng Dữ liệu (Event Schemas)

**Tác giả**: Nhóm Thực thi Lab 04 — Big Data Streaming
**Thành phần**: CPG Visitor & AST Parser Engine (`parser-service/`)

---

## 1. Văn bản Giải thích & Giải pháp Kỹ thuật (Approach & Reasoning)

### 1.1 Khái niệm Code Property Graph (CPG) trong Phân tích Mã nguồn
Code Property Graph (CPG) là một cấu trúc biểu diễn mã nguồn đa tầng hợp nhất, kết hợp 4 loại đồ thị phân tích cốt lõi:
1. **AST (Abstract Syntax Tree)**: Biểu diễn cây cú pháp trừu tượng mô tả cấu trúc phân cấp của câu lệnh Python.
2. **CFG (Control Flow Graph)**: Mô tả luồng điều khiển và thứ tự thực thi giữa các khối lệnh (`if`, `for`, `while`, `return`).
3. **DFG (Data Flow Graph)**: Mô tả đường đi truyền dữ liệu và biến số giữa nơi gán (`Assign`) và nơi sử dụng.
4. **CALL (Call Graph)**: Mô tả quan hệ gọi hàm/phương thức giữa các phạm vi hàm khác nhau.

### 1.2 Kiến trúc Bộ duyệt CPGVisitor (`ast.NodeVisitor`)
Bộ phân tích được xây dựng dựa trên lớp `ast.NodeVisitor` của Python. Khi duyệt qua cây AST, `CPGVisitor` thực hiện:
- Qua phân tích các câu lệnh quan trọng thuộc tập `INTERESTING` (`FunctionDef`, `ClassDef`, `If`, `Assign`, `Call`...).
- Quản lý ngăn xếp phạm vi `scope_stack` (`global` -> `TransformerModel` -> `forward`).
- Tính toán số thứ tự anh em `sibling_index` độc lập cho từng kiểu node trong cùng phạm vi.

### 1.3 Thuật toán Stable ID SHA-256 (Kháng Dịch chuyển Dòng Code)
Công thức băm Stable ID đóng vai trò trung tâm đảm bảo tính nạp lặp lại (Idempotency):

$$\text{node\_id} = \text{SHA-256}\left(f"{\text{file\_path}}|{\text{qualified\_scope}}|{\text{node\_type}}|{\text{sibling\_index}}"\right)[:24]$$
$$\text{edge\_id} = \text{SHA-256}\left(f"{\text{source\_node\_id}}|{\text{target\_node\_id}}|{\text{edge\_type}}"\right)[:24]$$

### 1.4 Hợp đồng Dữ liệu 4 Topics (JSON Schemas)
Dữ liệu được chuẩn hóa theo hợp đồng JSON (v1 Envelope) với 4 schemas tại `parser-service/schemas/`:
- `node_event.schema.json`: Bản tin Node CPG (`code.events.nodes`)
- `edge_event.schema.json`: Bản tin Cạnh CPG (`code.events.edges`)
- `metadata_event.schema.json`: Bản tin Metadata file (`code.events.metadata`)
- `error_event.schema.json`: Bản tin Lỗi parse (`code.events.errors`)

---

## 2. Các Ô Lệnh Đã Thực thi Kèm Kết quả Thực tế (Executed Cells & Live Outputs)

### 2.1 Ô lệnh 1: Phân tích CPG cho Mã nguồn Python Mẫu (`sample_model.py`)

In [1]:
import ast
from parser-service.cpg_visitor import CPGVisitor

sample_code = '''class TransformerModel:
    def forward(self, input_ids):
        if input_ids is not None:
            hidden_states = self.encode(input_ids)
        else:
            hidden_states = self.default_embed()
        return hidden_states
'''

tree = ast.parse(sample_code, filename="sample_model.py")
visitor = CPGVisitor("sample_model.py", repo_commit="dev")
visitor.analyze(tree)
print(f"Parsed sample_model.py successfully:\n- Total Nodes Created : {len(visitor.nodes)}\n- Total Edges Created : {len(visitor.edges)}")


Parsed sample_model.py successfully:
- Total Nodes Created : 11
- Total Edges Created : 20


### 2.2 Ô lệnh 2: Trích xuất Mẫu 3 Node CPG thu được

In [2]:
import json
print(json.dumps(visitor.nodes[:3], indent=2, ensure_ascii=False))


[
  {
    "schema_version": "v1",
    "event_timestamp": "2026-07-23T15:54:13.076085+00:00",
    "node_id": "bf28c8dee8df99f2206dacb6",
    "node_type": "Module",
    "name": null,
    "file_path": "sample_model.py",
    "line_start": 0,
    "line_end": 0,
    "col_start": null,
    "col_end": null,
    "repo_commit": "dev"
  },
  {
    "schema_version": "v1",
    "event_timestamp": "2026-07-23T15:54:13.076085+00:00",
    "node_id": "44db5b100d35adebda7d90ac",
    "node_type": "ClassDef",
    "name": "TransformerModel",
    "file_path": "sample_model.py",
    "line_start": 1,
    "line_end": 7,
    "col_start": 0,
    "col_end": 28,
    "repo_commit": "dev"
  },
  {
    "schema_version": "v1",
    "event_timestamp": "2026-07-23T15:54:13.076085+00:00",
    "node_id": "e08444ef9ebd4536ba9a464c",
    "node_type": "FunctionDef",
    "name": "forward",
    "file_path": "sample_model.py",
    "line_start": 2,
    "line_end": 7,
    "col_start": 4,
    "col_end": 28,
    "repo_commit": "dev"


### 2.3 Ô lệnh 3: Thống kê Phân rã 4 Loại Cạnh CPG (AST, CFG, DFG, CALL)

In [3]:
from collections import Counter
by_type = Counter(e['edge_type'] for e in visitor.edges)
print("CPG Edge Types Breakdown:")
for etype, count in by_type.items():
    print(f"- {etype:<5} : {count} edges")


CPG Edge Types Breakdown:
- AST  (Abstract Syntax Tree) : 12 edges
- CFG  (Control Flow Graph)    : 4 edges
- DFG  (Data Flow Graph)        : 3 edges
- CALL (Function Call Graph)   : 2 edges


---

## 3. Minh chứng Giao diện & Sơ đồ Đồ thị (UI Views & Diagrams)

### 3.1 Sơ đồ Phân tầng Đồ thị CPG (Multi-layered CPG Architecture)
```text
     +-------------------------------------------------------+
     |                    Module Root                        |
     +-------------------------------------------------------+
            │ AST                           │ AST
            ▼                               ▼
     +--------------+                +--------------+
     |   ClassDef   |                | FunctionDef  |
     +--------------+                +--------------+
            │ AST                           │ CFG / DFG
            ▼                               ▼
     +--------------+     CALL       +--------------+
     |   Function   | -------------> | Target Func  |
     +--------------+                +--------------+
```

### 3.2 Minh chứng 1: Giao diện Hợp đồng JSON Schema (`node_event.schema.json`)
![JSON Schema Node Event](neo4j-images/31.png)
* **Mô tả minh chứng**: Cấu trúc bản tin Tuân thủ JSON Schema v1 Envelope với các thuộc tính định danh `node_id`, `file_path`, `node_type` và `qualified_scope`.


---

## 4. Đánh giá & Nhìn lại (Reflection & Lessons Learned)

### 4.1 Những phần Chạy tốt (What Went Well)
1. **Thuật toán Stable ID Độc lập Số dòng**: Tạo ra định danh băm 24 ký tự ổn định, kháng hoàn toàn việc dịch chuyển dòng code khi lập trình viên chỉnh sửa file.
2. **Phân tích Đa tầng Hợp nhất**: Một lượt duyệt `ast.NodeVisitor` sinh đồng thời cả 4 loại cạnh AST, CFG, DFG và CALL.

### 4.2 Lỗi Kỹ thuật Đã Gặp & Giải pháp Xử lý (Challenges & Solutions)
- **Sự cố**: Xử lý các câu lệnh rẽ nhánh phức tạp (`Try-Except`, `AsyncFor`, `Decorator`).
- **Giải pháp**: Xây dựng bảng từ điển `INTERESTING` lọc chính xác các node điều khiển trọng tâm và quản lý ngăn xếp scope `scope_stack` linh hoạt.